> # ⏳ STATUS: PENDING FULL RUN
>
> **The cells below are SETUP, not final results.** The 22-slide stratified cohort is
> wired in and the download/comparison plumbing is in place, but the full 22-slide
> GrandQC run has **not been executed** (a multi-GB, GPU-scale job). Any saved outputs
> are cohort-selection/setup outputs, not cohort results.
>
> The validated headline numbers (99.957% / 99.988%) come from the **5-slide** set in
> `02_svs_dicom_validation.ipynb`. When this notebook is run at full n, export the
> table to `results/summary_tables/stratified_22_equivalence.csv` and update
> `docs/findings.md`.


# Is the IDC DICOM Route Lossless? — DICOM vs. Original SVS

**Companion study to `IDC_experiment.ipynb`.** Standalone: it runs the whole pipeline
itself and shares no state with the main notebook.

## The question

`IDC_experiment.ipynb` Part 6 compares GrandQC masks computed from **IDC DICOM** against
the **Zenodo reference masks** published by the GrandQC authors, and finds a gap on some
slides. It is tempting to read that gap as damage done by IDC's DICOM conversion.

That comparison cannot support the claim, because it varies **two things at once**:

1. the **input format** — IDC DICOM vs. the original Aperio `.svs`, and
2. the **reference provenance** — this pipeline vs. whatever produced the published masks.

## The design

Hold the pipeline fixed and vary only the format. The GrandQC reference masks were
computed from the original TCGA `.svs` files, which live in the **GDC** — a different
archive from IDC. The Zenodo mask filenames embed the source `.svs` filename, so each
reference mask can be resolved back to its exact source slide and fetched from GDC's
open-access API.

Running the **identical scripts, models and MPP** over both inputs gives a clean
three-way read:

| Comparison | What it isolates |
|---|---|
| our DICOM run **vs** our SVS run | **input format only** — same code, same models |
| Zenodo **vs** our DICOM run | format + reference provenance (confounded) |
| Zenodo **vs** our SVS run | reference provenance only |

If feeding the original `.svs` closes the Zenodo gap, the DICOM conversion is responsible.
If the gap survives unchanged, it is not.

> **Scope note.** The project charter puts *"integration with non-IDC or clinical /
> proprietary slide repositories"* out of scope. GDC is used here purely as a **measurement
> control** — the pipeline never ingests `.svs` in production, and nothing downstream
> depends on it. The control has to come from outside IDC, or it would be circular: you
> cannot validate IDC against itself.

> **Do not transcode the DICOM into `.svs` as a shortcut.** That would carry the DICOM's
> pixels into the "original" arm and the test would compare DICOM against itself,
> confirming nothing. The whole design rests on GDC holding an *independent* encoding of
> the same slide.


In [ ]:
# ── Environment setup (Colab-compatible) ─────────────────────────────────────
import sys, subprocess

# Build stamp. Colab keeps its OWN copy of this notebook (uploading copies it into
# Drive); it does not track the file on disk. If this banner is not what you expect,
# the Colab copy is stale -- re-upload the notebook.
NOTEBOOK_BUILD = "2026-07-16-B2"
NOTEBOOK_CONTENTS = "SVS validation study - DICOM vs original .svs, 22 stratified BRCA slides"
print(f"IDC_svs_validation build {NOTEBOOK_BUILD}")
print(f"  {NOTEBOOK_CONTENTS}")
print()

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab" if IN_COLAB else "Running locally")

if IN_COLAB:
    # libvips supplies the pyvips binary backend; it is not present in Colab by default.
    subprocess.run("apt-get -qq update && apt-get -qq install -y libvips libvips-tools",
                   shell=True, check=False)

# pydicom  -> read the total-pixel-matrix origin
# pyvips   -> canvas + OME-TIFF + dzsave
# tifffile -> memory-bounded streaming of the level-0 raster
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "idc-index", "wsidicom", "pydicom", "pyvips", "tifffile"], check=False)

try:
    import pyvips
    print(f"pyvips {pyvips.__version__}  (libvips {pyvips.base.version(0)}."
          f"{pyvips.base.version(1)}.{pyvips.base.version(2)})")
except OSError as exc:
    print(f"pyvips could not load libvips: {exc}\n"
          "Linux: apt install libvips  |  macOS: brew install vips  |  "
          "Windows: install the libvips binary and add its bin/ to PATH.")


## Part A: Environment, Cohort, and the DICOM Arm

Same setup as the main notebook: the same ten BRCA slides, the same GrandQC fork, the same
model weights. This section reproduces the DICOM arm so the study stands alone.


In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import IFrame, display
from idc_index import IDCClient

idc_client = IDCClient()
print(f"IDC data version: {idc_client.get_idc_version()}")

IDC data version: v24


In [ ]:
# The BRCA query below joins sm_index, which must be fetched into the local
# DuckDB before it can be referenced.
idc_client.fetch_index("sm_index")
print("sm_index loaded")


In [5]:
# Query TCGA-BRCA H&E slides with key metadata
tcga_brca_he = idc_client.sql_query("""
    SELECT
        i.SeriesInstanceUID,
        i.PatientID,
        s.ContainerIdentifier,
        s.primaryAnatomicStructureModifier_CodeMeaning AS tissue_type,
        s.max_TotalPixelMatrixColumns                  AS width_px,
        s.max_TotalPixelMatrixRows                     AS height_px,
        s.min_PixelSpacing_2sf                         AS pixel_spacing_mm,
        s.ObjectiveLensPower                           AS objective_power,
        ROUND(i.series_size_MB, 1)                     AS size_MB
    FROM index i
    JOIN sm_index s ON i.SeriesInstanceUID = s.SeriesInstanceUID
    WHERE i.collection_id = 'tcga_brca'
      AND array_to_string(s.staining_usingSubstance_CodeMeaning, ', ') LIKE '%hematoxylin%'
    ORDER BY i.series_size_MB ASC
""")

print(f"TCGA-BRCA H&E slides: {len(tcga_brca_he)}")
print(f"Tissue types: {tcga_brca_he['tissue_type'].value_counts().to_dict()}")
print(f"\nSize range: {tcga_brca_he['size_MB'].min()} – {tcga_brca_he['size_MB'].max()} MB")
tcga_brca_he.head(10)

TCGA-BRCA H&E slides: 3111
Tissue types: {'Neoplasm, Primary': 2704, 'Normal': 399}

Size range: 9.1 – 3586.3 MB


,SeriesInstanceUID,PatientID,ContainerIdentifier,tissue_type,width_px,height_px,pixel_spacing_mm,objective_power,size_MB
0,1.3.6.1.4.1.5962.99.1.1247079754.248369703.163...,TCGA-A7-A26J,TCGA-A7-A26J-01B-02-BS2,"Neoplasm, Primary",8618,7243,0.00025,40,9.1
1,1.3.6.1.4.1.5962.99.1.1279025007.575797294.163...,TCGA-AC-A2QJ,TCGA-AC-A2QJ-11A-02-TS2,Normal,33993,6923,0.00050,20,15.5
2,1.3.6.1.4.1.5962.99.1.1306193840.1223239355.16...,TCGA-AC-A2FB,TCGA-AC-A2FB-11A-01-TSA,Normal,31993,7690,0.00050,20,16.5
3,1.3.6.1.4.1.5962.99.1.1263739509.328647659.163...,TCGA-BH-A5IZ,TCGA-BH-A5IZ-11A-01-TS1,Normal,20041,15048,0.00050,20,18.7
4,1.3.6.1.4.1.5962.99.1.1290706954.176399971.163...,TCGA-AC-A2QJ,TCGA-AC-A2QJ-11A-01-TS1,Normal,41991,9316,0.00050,20,22.8
5,1.3.6.1.4.1.5962.99.1.1315471301.865934712.163...,TCGA-E9-A1RF,TCGA-E9-A1RF-11A-03-TSC,Normal,27888,9784,0.00050,20,23.0
6,1.3.6.1.4.1.5962.99.1.1322835661.1509329658.16...,TCGA-GM-A2D9,TCGA-GM-A2D9-11A-02-TS2,Normal,35992,8945,0.00050,20,25.0
7,1.3.6.1.4.1.5962.99.1.1255105493.445850269.163...,TCGA-E9-A1RF,TCGA-E9-A1RF-11A-01-TSA,Normal,23904,14929,0.00050,20,26.9
8,1.3.6.1.4.1.5962.99.1.1247939226.656219549.163...,TCGA-OL-A5RW,TCGA-OL-A5RW-01Z-00-DX1,"Neoplasm, Primary",12879,13618,0.00050,<NA>,27.1
9,1.3.6.1.4.1.5962.99.1.1244105102.1853648157.16...,TCGA-E9-A1NF,TCGA-E9-A1NF-11A-05-TSE,Normal,13943,17330,0.00049,20,27.3


### A.1 Stratified validation cohort (22 slides)

The equivalence test is only convincing if it holds across the axes that actually vary in
IDC. Three are queryable from the IDC index without downloading pixel data
(`transfer_syntax_name`, `Manufacturer`, `ObjectiveLensPower`); the fourth — the
photometric interpretation that distinguishes plain JPEG 2000 from the YBR_ICT variant
behind the OpenSlide color bug — is not in any index and was read from the DICOM headers
of the 39 JPEG 2000 slides via HTTP range requests (no full download).

That header sweep is itself a result. Of the 1,104 TCGA-BRCA diagnostic H&E slides with a
Zenodo reference mask, **29 are confirmed YBR_ICT (2.6%; up to 34 / 3.1% including five
that could not be re-read)** — spanning two source sites (AC at 40x, AO at 20x). Those are
exactly the slides an OpenSlide-based pipeline silently corrupts. The rest split into 5
JPEG 2000 slides that decode as RGB (the codec, without the bug) and ~1,065 plain JPEG.

The cohort below oversamples the rare-but-critical strata on purpose:

| Stratum | n | why |
|---|---|---|
| **YBR_ICT** | 8 | the bug-vulnerable codec; spans AC/40x and AO/20x |
| **JPEG 2000 → RGB** | 1 | J2K codec *without* YBR_ICT — the control that shows the codec alone is harmless |
| **JPEG Baseline → RGB** | 13 | the 96% bulk case, both scanners and magnifications |

> **Documented confound:** in TCGA-BRCA, codec, source-site and (partly) magnification
> travel together — nearly all YBR_ICT slides are site AC (40x) or AO (20x), and Zeiss
> slides are almost all plain JPEG. A fully orthogonal grid is therefore not achievable
> from this collection. This cohort is designed to demonstrate that DICOM = SVS holds
> *across* these strata, not to attribute cause to any one of them.


In [ ]:
VALIDATION_BARCODES = [
    # ---- YBR_ICT (8) ----
    "TCGA-AC-A23G-01Z-00-DX1",  # Leica 40x   43.7 MB
    "TCGA-AC-A23C-01Z-00-DX1",  # Leica 40x   56.6 MB
    "TCGA-AC-A62V-01Z-00-DX1",  # Leica 40x   59.7 MB
    "TCGA-AC-A2FO-01Z-00-DX1",  # Leica 40x   74.1 MB
    "TCGA-AC-A6IV-01Z-00-DX1",  # Leica 40x   78.3 MB
    "TCGA-AO-A1KQ-01Z-00-DX1",  # Leica 20x  250.4 MB
    "TCGA-AO-A1KP-01Z-00-DX1",  # Leica 20x  265.9 MB
    "TCGA-AO-A1KS-01Z-00-DX1",  # Leica 20x  330.3 MB
    # ---- J2K-RGB (1) ----
    "TCGA-A8-A0AB-01Z-00-DX1",  # Zeiss 40x  151.0 MB
    # ---- JPEG-RGB (13) ----
    "TCGA-MS-A51U-01Z-00-DX1",  # Leica 20x   76.2 MB
    "TCGA-AC-A3W5-01Z-00-DX1",  # Leica 40x   97.9 MB
    "TCGA-AN-A0XW-01Z-00-DX1",  # Leica 40x  108.1 MB
    "TCGA-A7-A6VX-01Z-00-DX2",  # Leica 40x  124.1 MB
    "TCGA-PL-A8LV-01A-01-DX1",  # Leica 40x  129.5 MB
    "TCGA-PL-A8LZ-01A-01-DX1",  # Leica 40x  135.0 MB
    "TCGA-A7-A13G-01Z-00-DX1",  # Leica 40x  137.0 MB
    "TCGA-AC-A3W6-01Z-00-DX1",  # Leica 20x  169.9 MB
    "TCGA-HN-A2NL-01Z-00-DX1",  # Leica 20x  180.5 MB
    "TCGA-OL-A66H-01Z-00-DX1",  # Zeiss 20x  248.1 MB
    "TCGA-A8-A081-01Z-00-DX1",  # Zeiss 40x  273.3 MB
    "TCGA-A8-A096-01Z-00-DX1",  # Zeiss 40x  287.7 MB
    "TCGA-A8-A09E-01Z-00-DX1",  # Zeiss 40x  304.2 MB
]

available = set(tcga_brca_he["ContainerIdentifier"])
present   = [b for b in VALIDATION_BARCODES if b in available]
missing   = [b for b in VALIDATION_BARCODES if b not in available]
if missing:
    print(f"WARNING: {len(missing)} barcode(s) not in this IDC version: {missing}")

demo_slides = (
    tcga_brca_he[tcga_brca_he["ContainerIdentifier"].isin(present)]
    .drop_duplicates(subset="ContainerIdentifier")
    .set_index("ContainerIdentifier")
    .loc[present]
    .reset_index()
)
print(f"Validation cohort: {len(demo_slides)} slides, "
      f"{demo_slides['size_MB'].sum():.0f} MB DICOM (+ a matching ~{demo_slides['size_MB'].sum()*1.3:.0f} MB "
      f"of .svs from GDC in Part B)")
display(demo_slides[["ContainerIdentifier", "width_px", "height_px",
                     "pixel_spacing_mm", "size_MB"]])


In [8]:
%%capture
# Clone the IDC-patched GrandQC fork (branch idc-dicom-wsidicom, shallow clone).
# This branch builds on idc-dicom-fixes (the fixes that let the inference scripts
# run directly on IDC DICOM series) and adds one further change: it reads DICOM
# slides through wsidicom instead of OpenSlide, working around an OpenSlide 4.0.1
# bug that mis-decodes YBR_ICT JPEG 2000 frames (see Part 2.3). Point the clone
# back at idc-dicom-fixes to restore the OpenSlide reader once that bug is fixed
# upstream.
!git clone --depth 1 --branch idc-dicom-wsidicom https://github.com/fedorov/grandqc.git

# Install GrandQC Python dependencies.
# PyTorch is already present in Colab; we install the remaining packages.
# segmentation-models-pytorch 0.3.1 is pinned because its model.predict() API
# was removed in 0.4.0 and GrandQC relies on it. wsidicom provides the DICOM
# reader used in place of OpenSlide.
!pip install -q \
    "segmentation-models-pytorch==0.3.1" \
    "rasterio>=1.3" \
    "imagecodecs" \
    "zarr<3" \
    "tifffile" \
    "wsidicom"

In [9]:
import os
import urllib.request

GRANDQC_SCRIPTS = "grandqc/01_WSI_inference_OPENSLIDE_QC"
MODELS_TD = os.path.join(GRANDQC_SCRIPTS, "models", "td")
MODELS_QC = os.path.join(GRANDQC_SCRIPTS, "models", "qc")
os.makedirs(MODELS_TD, exist_ok=True)
os.makedirs(MODELS_QC, exist_ok=True)

# (destination, source URL). Fetched with urllib.request so the cell runs the same
# on Windows, macOS and Linux — the shell wget used previously is not on Windows.
MODELS = [
    # Tissue detection model (~27 MB) — Zenodo record 14507273
    (os.path.join(MODELS_TD, "Tissue_Detection_MPP10.pth"),
     "https://zenodo.org/records/14507273/files/Tissue_Detection_MPP10.pth?download=1"),
    # Artifact segmentation model at MPP=1.5 (~25 MB) — Zenodo record 14041538.
    # MPP=1.5 (≈7× objective) is the recommended default for most scanners.
    (os.path.join(MODELS_QC, "GrandQC_MPP15.pth"),
     "https://zenodo.org/records/14041538/files/GrandQC_MPP15.pth?download=1"),
]

for dest, url in MODELS:
    # Skip a model that is already downloaded so re-running the cell is cheap.
    if os.path.exists(dest) and os.path.getsize(dest) > 1024 ** 2:
        print(f"Already present, skipping: {dest}")
        continue
    print(f"Downloading {os.path.basename(dest)} ...")
    urllib.request.urlretrieve(url, dest)

print()
print("Models downloaded:")
for dest, _ in MODELS:
    size_mb = os.path.getsize(dest) / 1024 ** 2
    print(f"  {dest}  ({size_mb:.1f} MB)")

Already present, skipping: grandqc/01_WSI_inference_OPENSLIDE_QC\models\td\Tissue_Detection_MPP10.pth
Already present, skipping: grandqc/01_WSI_inference_OPENSLIDE_QC\models\qc\GrandQC_MPP15.pth

Models downloaded:
  grandqc/01_WSI_inference_OPENSLIDE_QC\models\td\Tissue_Detection_MPP10.pth  (25.4 MB)
  grandqc/01_WSI_inference_OPENSLIDE_QC\models\qc\GrandQC_MPP15.pth  (24.2 MB)


In [11]:
SLIDE_DIR = "slides"
os.makedirs(SLIDE_DIR, exist_ok=True)

print(f"Downloading {len(demo_slides)} slides  →  {SLIDE_DIR}/")
print("(This may take a few minutes depending on file sizes and network speed)\n")

idc_client.download_from_selection(
    downloadDir=SLIDE_DIR,
    seriesInstanceUID=demo_slides["SeriesInstanceUID"].tolist(),
    dirTemplate="%SeriesInstanceUID",
)
print("\nDownload complete.")

2026-07-12 21:50:47,042 - Disk size needed: 387.21 MB
2026-07-12 21:50:47,043 - Disk size available: 717.74 GB
2026-07-12 21:50:47,098 - Not using s5cmd sync as the destination folder is empty or sync or progress bar is not requested
2026-07-12 21:50:47,112 - Initial size of the directory: 0 bytes
2026-07-12 21:50:47,113 - Approximate size of the files that need to be downloaded: 387.21 MB


(This may take a few minutes depending on file sizes and network speed)



2026-07-12 21:51:06,281 - Successfully downloaded files to C:\Users\ronak\IDC-Tutorials\notebooks\pathomics\slides



Download complete.


In [12]:
import shutil

# Build SeriesInstanceUID → ContainerIdentifier map
uid_to_barcode = dict(zip(demo_slides["SeriesInstanceUID"], demo_slides["ContainerIdentifier"]))

# Rename each downloaded UID directory to its barcode. This is written to be
# safe to re-run, and to tolerate the download cell being re-run (which would
# re-create the UID directory next to an already-renamed barcode directory).
for uid, barcode in uid_to_barcode.items():
    src = os.path.join(SLIDE_DIR, uid)      # freshly downloaded, named by UID
    dst = os.path.join(SLIDE_DIR, barcode)  # renamed target, named by barcode

    if os.path.isdir(src) and os.path.isdir(dst):
        # Both present → the download cell was re-run after a previous rename.
        # The UID directory is a redundant copy of the same (immutable) series,
        # so drop it and keep the barcode directory downstream code expects.
        shutil.rmtree(src)
        print(f"  Removed duplicate UID dir; kept {barcode}")
    elif os.path.isdir(src):
        os.rename(src, dst)
        print(f"  {uid}  →  {barcode}")
    elif os.path.isdir(dst):
        print(f"  Already renamed: {barcode}")
    else:
        print(f"  WARNING: directory not found: {uid}")

print("\nSlide directories after renaming:")
for d in sorted(os.listdir(SLIDE_DIR)):
    dpath = os.path.join(SLIDE_DIR, d)
    if os.path.isdir(dpath):
        n = sum(1 for f in os.listdir(dpath) if f.lower().endswith(".dcm"))
        print(f"  {d}/  ({n} .dcm files)")

  Removed duplicate UID dir; kept TCGA-AC-A23G-01Z-00-DX1
  Removed duplicate UID dir; kept TCGA-AC-A23C-01Z-00-DX1
  Removed duplicate UID dir; kept TCGA-AC-A62V-01Z-00-DX1
  Removed duplicate UID dir; kept TCGA-MS-A51U-01Z-00-DX1
  Removed duplicate UID dir; kept TCGA-A8-A0AB-01Z-00-DX1

Slide directories after renaming:
  TCGA-A8-A0AB-01Z-00-DX1/  (4 .dcm files)
  TCGA-AC-A23C-01Z-00-DX1/  (4 .dcm files)
  TCGA-AC-A23G-01Z-00-DX1/  (4 .dcm files)
  TCGA-AC-A62V-01Z-00-DX1/  (4 .dcm files)
  TCGA-MS-A51U-01Z-00-DX1/  (4 .dcm files)


In [13]:
import wsidicom
from wsidicom import WsiDicom

print(f"wsidicom version: {wsidicom.__version__}\n")

for barcode in uid_to_barcode.values():
    slide_dir = os.path.join(SLIDE_DIR, barcode)
    dcm_files = sorted(f for f in os.listdir(slide_dir) if f.lower().endswith(".dcm"))
    if not dcm_files:
        print(f"  {barcode}: no .dcm files found")
        continue
    try:
        slide = WsiDicom.open(slide_dir)
        w, h = slide.size.width, slide.size.height
        mpp = round(slide.mpp.width, 4) if slide.mpp else "N/A"
        vendor = getattr(slide.metadata.equipment, "manufacturer", None) or "N/A"
        print(f"  {barcode}")
        print(f"    {w:,} × {h:,} px  |  MPP={mpp}  |  vendor={vendor}  |  levels={len(slide.levels)}")
        slide.close()
    except Exception as exc:
        print(f"  {barcode}: ERROR — {exc}")

print("\nAll slides verified.")

2026-07-12 21:51:06,854 - get_frame_offsets is deprecated and will be removed in v4.0


wsidicom version: 0.33.1

  TCGA-AC-A23G-01Z-00-DX1
    19,920 × 18,542 px  |  MPP=0.2457  |  vendor=Leica Biosystems  |  levels=3
  TCGA-AC-A23C-01Z-00-DX1
    19,920 × 26,420 px  |  MPP=0.2457  |  vendor=Leica Biosystems  |  levels=3
  TCGA-AC-A62V-01Z-00-DX1
    37,410 × 18,719 px  |  MPP=0.252  |  vendor=Leica Biosystems  |  levels=3
  TCGA-MS-A51U-01Z-00-DX1
    20,000 × 23,112 px  |  MPP=0.499  |  vendor=Leica Biosystems  |  levels=3
  TCGA-A8-A0AB-01Z-00-DX1
    35,584 × 42,752 px  |  MPP=0.2325  |  vendor=Carl Zeiss  |  levels=3

All slides verified.


In [14]:
import subprocess, sys

OUTPUT_DIR = "qc_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

slide_dir_abs  = os.path.abspath(SLIDE_DIR)
output_dir_abs = os.path.abspath(OUTPUT_DIR)
scripts_abs    = os.path.abspath(GRANDQC_SCRIPTS)

print(f"Slides:          {slide_dir_abs}")
print(f"Output:          {output_dir_abs}")
print(f"Scripts dir:     {scripts_abs}")
print()
print("Running wsi_tis_detect.py ...")

result = subprocess.run(
    [sys.executable, "wsi_tis_detect.py",
     "--slide_folder", slide_dir_abs,
     "--output_dir",   output_dir_abs],
    cwd=scripts_abs,
)

if result.returncode != 0:
    print(f"\nERROR: wsi_tis_detect.py exited with code {result.returncode}")
else:
    mask_dir = os.path.join(output_dir_abs, "tis_det_mask")
    masks = sorted(os.listdir(mask_dir))
    print(f"\nTissue detection complete — {len(masks)} mask(s) written to {mask_dir}/")
    for m in masks:
        print(f"  {m}")

Slides:          c:\Users\ronak\IDC-Tutorials\notebooks\pathomics\slides
Output:          c:\Users\ronak\IDC-Tutorials\notebooks\pathomics\qc_output
Scripts dir:     c:\Users\ronak\IDC-Tutorials\notebooks\pathomics\grandqc\01_WSI_inference_OPENSLIDE_QC

Running wsi_tis_detect.py ...

Tissue detection complete — 5 mask(s) written to c:\Users\ronak\IDC-Tutorials\notebooks\pathomics\qc_output\tis_det_mask/
  TCGA-A8-A0AB-01Z-00-DX1_MASK.png
  TCGA-AC-A23C-01Z-00-DX1_MASK.png
  TCGA-AC-A23G-01Z-00-DX1_MASK.png
  TCGA-AC-A62V-01Z-00-DX1_MASK.png
  TCGA-MS-A51U-01Z-00-DX1_MASK.png


In [15]:
print("Running main.py (artifact segmentation) ...")

result = subprocess.run(
    [sys.executable, "main.py",
     "--slide_folder", slide_dir_abs,
     "--output_dir",   output_dir_abs,
     "--mpp_model",    "1.5",
     "--create_geojson", "N"],
    cwd=scripts_abs,
)

if result.returncode != 0:
    print(f"\nERROR: main.py exited with code {result.returncode}")
else:
    maps_dir = os.path.join(output_dir_abs, "maps_qc")
    maps = sorted(os.listdir(maps_dir))
    print(f"\nArtifact segmentation complete — {len(maps)} map(s) in {maps_dir}/")
    for m in maps:
        print(f"  {m}")

    # Locate the TSV report (name includes the slide count)
    reports = [f for f in os.listdir(output_dir_abs) if f.endswith("_stats_per_slide.txt")]
    if reports:
        import pandas as pd
        report_path = os.path.join(output_dir_abs, reports[0])
        df = pd.read_csv(report_path, sep="\t")
        print(f"\nPer-slide report ({reports[0]}):")
        display(df[["slide_name", "mpp", "height", "width", "time"]])

Running main.py (artifact segmentation) ...

Artifact segmentation complete — 5 map(s) in c:\Users\ronak\IDC-Tutorials\notebooks\pathomics\qc_output\maps_qc/
  TCGA-A8-A0AB-01Z-00-DX1_map_QC.png
  TCGA-AC-A23C-01Z-00-DX1_map_QC.png
  TCGA-AC-A23G-01Z-00-DX1_map_QC.png
  TCGA-AC-A62V-01Z-00-DX1_map_QC.png
  TCGA-MS-A51U-01Z-00-DX1_map_QC.png

Per-slide report (report_qc_output_0_5_stats_per_slide.txt):


,slide_name,mpp,height,width,time
0,TCGA-A8-A0AB-01Z-00-DX1,0.2325,39636,33030,1.0
1,TCGA-AC-A23C-01Z-00-DX1,0.2457,25000,18750,0.7
2,TCGA-AC-A23G-01Z-00-DX1,0.2457,15625,18750,0.3
3,TCGA-AC-A62V-01Z-00-DX1,0.252,18282,36564,0.8
4,TCGA-MS-A51U-01Z-00-DX1,0.499,23085,18468,0.8
5,slide_name,mpp,height,width,time
6,TCGA-A8-A0AB-01Z-00-DX1,0.2325,39636,33030,1.0
7,TCGA-AC-A23C-01Z-00-DX1,0.2457,25000,18750,0.6
8,TCGA-AC-A23G-01Z-00-DX1,0.2457,15625,18750,0.3
9,TCGA-AC-A62V-01Z-00-DX1,0.252,18282,36564,0.8


### A.2 Reference masks (Zenodo)

The GrandQC authors published pre-computed artifact masks for TCGA on Zenodo
(record 14041578), one `.tar` per cohort. `BRCA.tar` (~1.95 GB) holds masks for **1,105**
TCGA-BRCA diagnostic slides — 1,104 of the 1,133 BRCA DX slides in IDC — named
`BRCA/mask_qc/<barcode>.<UUID>.svs_mask.png`.

We download the archive once and extract only our ten slides.

> **The download is resumable on purpose.** Zenodo drops long transfers; a plain
> `urlretrieve` of a 2 GB file fails partway often enough that it is not safe to rely on.
> The loop below resumes from wherever it stopped via HTTP `Range`, which Zenodo honours
> even though it does not advertise `Accept-Ranges`.


In [ ]:
# ── GrandQC reference masks (Zenodo record 14041578) ─────────────────────────
import tarfile
import time

import requests

REF_MASKS_DIR = "brca_ref_masks"
os.makedirs(REF_MASKS_DIR, exist_ok=True)
BRCA_TAR = "BRCA.tar"
BRCA_URL = "https://zenodo.org/records/14041578/files/BRCA.tar?download=1"

total = int(requests.head(BRCA_URL, allow_redirects=True, timeout=60).headers["content-length"])
for attempt in range(1, 21):
    have = os.path.getsize(BRCA_TAR) if os.path.exists(BRCA_TAR) else 0
    if have >= total:
        break
    print(f"  BRCA.tar: {have / 1024**3:5.2f} / {total / 1024**3:.2f} GB ...", flush=True)
    try:
        r = requests.get(BRCA_URL, headers={"Range": f"bytes={have}-"},
                         stream=True, timeout=(30, 120))
        if r.status_code not in (200, 206):
            print(f"    unexpected status {r.status_code}; retrying")
            time.sleep(5)
            continue
        with open(BRCA_TAR, "ab") as f:
            for chunk in r.iter_content(1 << 20):
                if chunk:
                    f.write(chunk)
    except Exception as exc:                      # noqa: BLE001 - any drop is retryable
        print(f"    connection dropped ({type(exc).__name__}); resuming")
        time.sleep(3)

size = os.path.getsize(BRCA_TAR)
assert size == total, f"BRCA.tar incomplete: {size} of {total} bytes"
print(f"  BRCA.tar complete ({size / 1024**3:.2f} GB)\n")

# Extract only the masks for our cohort.
wanted = set(demo_slides["ContainerIdentifier"])
ref_mask_by_barcode = {}
with tarfile.open(BRCA_TAR) as tf:
    for member in tf:
        if not member.name.endswith("_mask.png"):
            continue
        base = os.path.basename(member.name)
        barcode = base.split(".")[0]              # '<barcode>.<UUID>.svs_mask.png'
        if barcode in wanted:
            with tf.extractfile(member) as src, open(os.path.join(REF_MASKS_DIR, base), "wb") as dst:
                dst.write(src.read())
            ref_mask_by_barcode[barcode] = base

print(f"Extracted {len(ref_mask_by_barcode)} of {len(wanted)} reference masks:")
for bc in sorted(ref_mask_by_barcode):
    print(f"  {bc}")
absent = wanted - set(ref_mask_by_barcode)
if absent:
    print(f"\nWARNING: no reference mask for {sorted(absent)} -- excluded from Part 6.")


### A.3 Comparison helpers and the Zenodo baseline

Three panels per slide: the reference (Zenodo, computed from the original `.svs`), our
local DICOM result, and an explicit **difference** map. Two artifact maps side by side
make it hard to see *where* they disagree, so the third panel classifies every pixel:

- **red** — tissue only our run found (the reference calls it background)
- **blue** — tissue only the reference found
- **amber** — both call it tissue but assign different artifact classes

That separation matters, because a tissue-detection boundary difference and an
artifact-classification difference are very different things. Panels are cropped to the
tissue, and each row reports whole-image agreement alongside agreement *within tissue
both pipelines detected* — the number that isolates the artifact model from the
tissue-detection boundary.

GrandQC mask values are 1-based (`1`=clean tissue … `7`=background). Value `0` is the
unwritten right/bottom margin left by the floor-grid — it is not a class, so it is drawn
as background here and excluded from every metric.


In [ ]:
from matplotlib.patches import Patch
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

# GrandQC palette. Mask values are 1-based; 0 is the unwritten margin, not a class.
QC_PAL = {
    1: ("#9e9e9e", "Clean tissue"),
    2: ("#ff6347", "Folds"),
    3: ("#00c853", "Dark spots"),
    4: ("#e53935", "Pen marks"),
    5: ("#ff00ff", "Bubbles/edges"),
    6: ("#4b0082", "Out-of-focus"),
    7: ("#ffffff", "Background"),
}


def qc_tissue(m):
    """Boolean tissue mask: any class except background (7) and the margin (0)."""
    return (m != 7) & (m != 0)


def qc_colorize(m):
    out = np.full(m.shape + (3,), 255, np.uint8)
    for v, (hx, _) in QC_PAL.items():
        out[m == v] = [int(hx[i:i + 2], 16) for i in (1, 3, 5)]
    out[m == 0] = 255
    return out


def qc_crop(a, b):
    """Common shape + a crop box around the tissue of either mask."""
    h, w = min(a.shape[0], b.shape[0]), min(a.shape[1], b.shape[1])
    a, b = a[:h, :w], b[:h, :w]
    ys, xs = np.where(qc_tissue(a) | qc_tissue(b))
    if len(ys) == 0:
        return a, b, (slice(0, h), slice(0, w))
    p = 40
    return a, b, (slice(max(0, ys.min() - p), min(h, ys.max() + p)),
                  slice(max(0, xs.min() - p), min(w, xs.max() + p)))


def qc_agreement(a, b):
    """(whole-image %, within-shared-tissue %) for two same-shaped masks."""
    h, w = min(a.shape[0], b.shape[0]), min(a.shape[1], b.shape[1])
    a, b = a[:h, :w], b[:h, :w]
    both = qc_tissue(a) & qc_tissue(b)
    inner = (a[both] == b[both]).mean() * 100 if both.any() else float("nan")
    return (a == b).mean() * 100, inner


def qc_compare_figure(pairs, col_titles, suptitle, subtitle=""):
    """pairs: list of (row_label, mask_A, mask_B). Renders A | B | difference."""
    fig, axes = plt.subplots(len(pairs), 3, figsize=(13.5, 3.05 * len(pairs)))
    axes = np.atleast_2d(axes)
    for r, (label, A, B) in enumerate(pairs):
        A, B, sl = qc_crop(A, B)
        at, bt = qc_tissue(A), qc_tissue(B)
        rim, lost, both = bt & ~at, at & ~bt, at & bt
        clash = both & (A != B)
        whole, inner = qc_agreement(A, B)

        diff = np.full((sl[0].stop - sl[0].start, sl[1].stop - sl[1].start, 3), 255, np.uint8)
        diff[both[sl]] = (232, 232, 232)
        diff[rim[sl]] = (255, 0, 0)
        diff[lost[sl]] = (0, 90, 255)
        diff[clash[sl]] = (255, 190, 0)

        for c, img in enumerate([qc_colorize(A[sl]), qc_colorize(B[sl]), diff]):
            ax = axes[r, c]
            ax.imshow(img, interpolation="nearest")
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(col_titles[c], fontsize=11, pad=8)
        axes[r, 0].set_ylabel(label, fontsize=8.5, labelpad=8)
        axes[r, 2].text(1.02, 0.5,
                        f"whole-image\n{whole:.2f}%\n\nwithin shared\ntissue\n{inner:.3f}%\n\n"
                        f"rim {rim.mean()*100:.2f}%\nclash {clash.mean()*100:.3f}%",
                        transform=axes[r, 2].transAxes, va="center", ha="left",
                        fontsize=8.5, family="monospace")

    fig.suptitle(suptitle, fontsize=14, fontweight="bold", y=0.997)
    if subtitle:
        fig.text(0.5, 0.978, subtitle, ha="center", fontsize=9.5, color="#555555")
    fig.legend(handles=[Patch(facecolor=h, edgecolor="#999999", label=n)
                        for _, (h, n) in QC_PAL.items()],
               loc="lower center", bbox_to_anchor=(0.5, 0.037), ncol=7, frameon=False,
               fontsize=9, title="Artifact classes (panels 1-2)", title_fontsize=9)
    fig.legend(handles=[
        Patch(facecolor="#e8e8e8", edgecolor="#999999", label="Both agree"),
        Patch(facecolor="#ff0000", label=f"Tissue only {col_titles[1].lower()} found"),
        Patch(facecolor="#005aff", label=f"Tissue only {col_titles[0].lower()} found"),
        Patch(facecolor="#ffbe00", label="Class disagreement in shared tissue")],
        loc="lower center", bbox_to_anchor=(0.5, 0.002), ncol=4, frameon=False,
        fontsize=9, title="Difference panel", title_fontsize=9)
    fig.tight_layout(rect=[0.015, 0.075, 0.945, 0.972])
    return fig


# Photometric interpretation predicts which slides behave differently, so label it.
def dicom_photometric(barcode):
    import pydicom
    best = None
    for f in sorted(glob.glob(os.path.join(SLIDE_DIR, barcode, "*.dcm"))):
        ds = pydicom.dcmread(f, stop_before_pixels=True)
        if "VOLUME" not in [str(v).upper() for v in getattr(ds, "ImageType", [])]:
            continue
        cols = int(getattr(ds, "TotalPixelMatrixColumns", 0) or 0)
        if best is None or cols > best[0]:
            best = (cols, str(ds.PhotometricInterpretation))
    return best[1] if best else "?"


import glob

pairs = []
for barcode in uid_to_barcode.values():
    ref_f = ref_mask_by_barcode.get(barcode)
    loc_p = os.path.join(OUTPUT_DIR, "mask_qc", f"{barcode}_mask.png")
    if not ref_f or not os.path.exists(loc_p):
        print(f"  skip {barcode} (missing mask)")
        continue
    pairs.append((f"{barcode}\n{dicom_photometric(barcode)}",
                  np.array(Image.open(os.path.join(REF_MASKS_DIR, ref_f))),
                  np.array(Image.open(loc_p))))

qc_compare_figure(
    pairs,
    ["Reference (Zenodo, from .svs)", "Your run (IDC DICOM)", "Difference"],
    "GrandQC artifact maps — Zenodo reference vs IDC DICOM pipeline",
    "Differences concentrate at the tissue-detection boundary, not in the artifact classes.",
)
plt.show()


## Part B: The SVS Arm

Now the same pipeline over the original `.svs` files from GDC.

This is the only part of either notebook that needs **OpenSlide** — the DICOM path never
touches it. GrandQC's `open_slide()` wrapper dispatches on the input: DICOM series go to
wsidicom, everything else falls back to OpenSlide.


In [ ]:
# ── OpenSlide: needed only to read .svs (the DICOM arm never uses it) ────────
if IN_COLAB:
    subprocess.run("apt-get -qq install -y openslide-tools", shell=True, check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "openslide-python", "openslide-bin"], check=False)

try:
    import openslide
    print(f"openslide-python {openslide.__version__} | "
          f"libopenslide {openslide.__library_version__}")
except Exception as exc:                          # noqa: BLE001
    print(f"OpenSlide unavailable ({exc}) - Part B cannot run without it.")


In [ ]:
# ── Resolve each reference mask back to its source .svs and fetch it from GDC ──
import json
import time
import urllib.parse
import urllib.request

SVS_DIR = "svs_slides"
os.makedirs(SVS_DIR, exist_ok=True)


def gdc_lookup(svs_filename):
    """GDC file record for an exact TCGA .svs filename (open access, no auth).

    The UUID inside the filename is a legacy TCGA identifier, NOT the GDC file_id --
    looking it up directly 404s. Query by file_name and let GDC return the real id.
    """
    filt = {"op": "=", "content": {"field": "file_name", "value": svs_filename}}
    q = ("https://api.gdc.cancer.gov/files?filters="
         + urllib.parse.quote(json.dumps(filt))
         + "&fields=file_id,file_size,access")
    hits = json.loads(urllib.request.urlopen(q, timeout=60).read())["data"]["hits"]
    return hits[0] if hits else None


print(f"Fetching original .svs for {len(ref_mask_by_barcode)} slide(s) -> {SVS_DIR}/\n")
for barcode, mask_fname in sorted(ref_mask_by_barcode.items()):
    svs_name = mask_fname[: -len("_mask.png")]     # '<barcode>.<UUID>.svs_mask.png'
    dest = os.path.join(SVS_DIR, f"{barcode}.svs")
    if os.path.exists(dest) and os.path.getsize(dest) > 1024 ** 2:
        print(f"  already present: {barcode} ({os.path.getsize(dest)/1024**2:.0f} MB)")
        continue
    hit = gdc_lookup(svs_name)
    if hit is None:
        print(f"  {barcode}: NOT FOUND on GDC ({svs_name})")
        continue
    t = time.time()
    urllib.request.urlretrieve("https://api.gdc.cancer.gov/data/" + hit["file_id"], dest)
    print(f"  {barcode}: {os.path.getsize(dest)/1024**2:6.0f} MB in {time.time()-t:4.0f}s")

total = sum(os.path.getsize(os.path.join(SVS_DIR, f)) for f in os.listdir(SVS_DIR))
print(f"\nTotal: {total / 1024**3:.2f} GB")


In [ ]:
# ── Run the SAME two GrandQC steps on the .svs files ──────────────────────────
# Identical scripts, models and MPP as the DICOM arm. Only the input format differs.
SVS_OUT = "qc_output_svs"
os.makedirs(SVS_OUT, exist_ok=True)
svs_dir_abs, svs_out_abs = os.path.abspath(SVS_DIR), os.path.abspath(SVS_OUT)

for step, args in [
    ("wsi_tis_detect.py", ["--slide_folder", svs_dir_abs, "--output_dir", svs_out_abs]),
    ("main.py", ["--slide_folder", svs_dir_abs, "--output_dir", svs_out_abs,
                 "--mpp_model", "1.5", "--create_geojson", "N"]),
]:
    print(f"Running {step} on .svs ...")
    r = subprocess.run([sys.executable, step] + args, cwd=scripts_abs)
    if r.returncode != 0:
        print(f"  ERROR: {step} exited {r.returncode}")
        break
else:
    masks = sorted(os.listdir(os.path.join(SVS_OUT, "mask_qc")))
    print(f"\nSVS run complete - {len(masks)} mask(s)")


## Part C: The Result

`DICOM vs SVS` is the number that matters: code, models and MPP held fixed, only the input
format varied. `SVS closes (pp)` tests the suspicion directly — if the DICOM conversion
caused the Zenodo gap, feeding the original `.svs` would close it.


In [ ]:
# ── Three-way comparison ──────────────────────────────────────────────────────
rows = []
for barcode in sorted(ref_mask_by_barcode):
    p_zen = os.path.join(REF_MASKS_DIR, ref_mask_by_barcode[barcode])
    p_dcm = os.path.join(OUTPUT_DIR, "mask_qc", f"{barcode}_mask.png")
    p_svs = os.path.join(SVS_OUT, "mask_qc", f"{barcode}.svs_mask.png")
    if not all(os.path.exists(p) for p in (p_zen, p_dcm, p_svs)):
        print(f"  skip {barcode} (missing a mask)")
        continue
    zen, dcm, svs = (np.array(Image.open(p)) for p in (p_zen, p_dcm, p_svs))
    dw, di = qc_agreement(svs, dcm)      # input format only
    zd, _ = qc_agreement(zen, dcm)
    zs, _ = qc_agreement(zen, svs)
    rows.append({
        "slide": barcode,
        "photometric": dicom_photometric(barcode),
        "DICOM vs SVS (whole)": round(dw, 3),
        "DICOM vs SVS (tissue)": round(di, 3),
        "Zenodo vs DICOM": round(zd, 3),
        "Zenodo vs SVS": round(zs, 3),
        "SVS closes (pp)": round(zs - zd, 3),
    })

df_svs = pd.DataFrame(rows)
display(df_svs)

if len(df_svs):
    print(f"\nDICOM reproduces the SVS route at "
          f"{df_svs['DICOM vs SVS (whole)'].mean():.3f}% whole-image / "
          f"{df_svs['DICOM vs SVS (tissue)'].mean():.3f}% within shared tissue.")
    print(f"Feeding the original .svs closes only "
          f"{df_svs['SVS closes (pp)'].mean():+.3f} pp of the Zenodo gap on average.")


In [ ]:
# ── Our two runs, side by side ────────────────────────────────────────────────
# If the DICOM conversion caused the boundary rim, this figure would show it.
# A near-empty difference column means it did not.
pairs = []
for barcode in sorted(ref_mask_by_barcode):
    p_dcm = os.path.join(OUTPUT_DIR, "mask_qc", f"{barcode}_mask.png")
    p_svs = os.path.join(SVS_OUT, "mask_qc", f"{barcode}.svs_mask.png")
    if not (os.path.exists(p_dcm) and os.path.exists(p_svs)):
        continue
    pairs.append((f"{barcode}\n{dicom_photometric(barcode)}",
                  np.array(Image.open(p_svs)), np.array(Image.open(p_dcm))))

if pairs:
    qc_compare_figure(
        pairs,
        ["Our run (original .svs)", "Our run (IDC DICOM)", "Difference"],
        "Same pipeline, same models - only the input format differs",
        "A near-empty difference column means switching SVS -> IDC DICOM is lossless.",
    )
    plt.show()


## What this establishes

Both arms land the **same distance** from the Zenodo reference. Feeding the original
`.svs` — the exact file the reference was computed from — does not close the gap.

Two conclusions follow:

1. **The IDC DICOM conversion is lossless for this pipeline.** Switching SVS → IDC DICOM
   changes the QC masks by a margin indistinguishable from noise. This is the claim a
   DICOM-based QC pipeline needs to make, and it is now measured rather than assumed.
2. **The residual gap to the reference is a property of the reference, not of IDC.** It
   reproduces identically from the original `.svs`, so it cannot be caused by anything the
   DICOM contains — nor fixed by anything the DICOM contains. The most likely explanation
   is that the published masks were produced by a different tissue-detection model version
   or settings than those this notebook downloads (Zenodo record 14507273); the published
   masks do not pin this down. **That remains a hypothesis, not a finding.**

The practical consequence for the main pipeline: **validate the DICOM route against the
SVS route, not against the Zenodo masks.** The former is a controlled comparison. The
latter compares against a third-party artifact of unknown provenance and silently
attributes any difference to the format under test — which is precisely the error this
notebook exists to rule out.
